In [1]:
#1.LAG() - current row se peeche(previous row) ki value laata hai - jaise "pichhla order kab tha".

In [2]:
#2.LEAD()- curerent row se aage(next row) ki value laata hai - jaise "agla order kab hoga".

In [3]:
#yeh dono recency calculation aur customer behaviour trend samjhne ke liye bahut jaruri hai - Bouns day35B(churn label banane) ke liye foundation  yahi banta hai.

In [4]:
#01:LAG() - har customer ke liye pichle order ki date nikalna:

In [5]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT "Customer ID",Invoice, InvoiceDate,
            LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate) as previous_order_date
    FROM orders
    ORDER BY "Customer ID",InvoiceDate
    LIMIT 30
""",conn)
print(q1)

    Customer ID  Invoice          InvoiceDate  previous_order_date
0       12346.0   491725  2009-12-14 08:34:00                  NaN
1       12346.0   491742  2009-12-14 11:00:00  2009-12-14 08:34:00
2       12346.0   491744  2009-12-14 11:02:00  2009-12-14 11:00:00
3       12346.0   492718  2009-12-18 10:47:00  2009-12-14 11:02:00
4       12346.0   492722  2009-12-18 10:55:00  2009-12-18 10:47:00
5       12346.0   493410  2010-01-04 09:24:00  2009-12-18 10:55:00
6       12346.0   493412  2010-01-04 09:53:00  2010-01-04 09:24:00
7       12346.0   494450  2010-01-14 13:50:00  2010-01-04 09:53:00
8       12346.0   495295  2010-01-22 13:30:00  2010-01-14 13:50:00
9       12346.0  C495800  2010-01-26 17:27:00  2010-01-22 13:30:00
10      12346.0   499763  2010-03-02 13:08:00  2010-01-26 17:27:00
11      12346.0   513774  2010-06-28 13:53:00  2010-03-02 13:08:00
12      12346.0  C514024  2010-06-30 11:22:00  2010-06-28 13:53:00
13      12346.0  C525099  2010-10-04 09:54:00  2010-06-30 11:2

In [6]:
#02:Gap between orders nikalna(days) - yeh Recency/churn logic ka core hai.

In [8]:
q2 = pd.read_sql("""
    SELECT "Customer ID", Invoice, InvoiceDate,
            LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate) as previous_order_date,
            julianday(InvoiceDate) - julianday(LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate)) as days_since_last_order
    FROM orders
    ORDER BY "Customer ID",InvoiceDate
    LIMIT 30
""",conn)
print(q2)

    Customer ID  Invoice          InvoiceDate  previous_order_date  \
0       12346.0   491725  2009-12-14 08:34:00                  NaN   
1       12346.0   491742  2009-12-14 11:00:00  2009-12-14 08:34:00   
2       12346.0   491744  2009-12-14 11:02:00  2009-12-14 11:00:00   
3       12346.0   492718  2009-12-18 10:47:00  2009-12-14 11:02:00   
4       12346.0   492722  2009-12-18 10:55:00  2009-12-18 10:47:00   
5       12346.0   493410  2010-01-04 09:24:00  2009-12-18 10:55:00   
6       12346.0   493412  2010-01-04 09:53:00  2010-01-04 09:24:00   
7       12346.0   494450  2010-01-14 13:50:00  2010-01-04 09:53:00   
8       12346.0   495295  2010-01-22 13:30:00  2010-01-14 13:50:00   
9       12346.0  C495800  2010-01-26 17:27:00  2010-01-22 13:30:00   
10      12346.0   499763  2010-03-02 13:08:00  2010-01-26 17:27:00   
11      12346.0   513774  2010-06-28 13:53:00  2010-03-02 13:08:00   
12      12346.0  C514024  2010-06-30 11:22:00  2010-06-28 13:53:00   
13      12346.0  C52

In [9]:
#julianday() SQLite ka function hai jo date ko number me convert karta hai, taki subtraction se din ka difference nikal sake.

In [10]:
#03:LEAD() - agle order ki date nikaalna(reverse direction):

In [11]:
q3 = pd.read_sql("""
    SELECT "Customer ID", Invoice,InvoiceDate,
            LEAD(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate) as next_order_date
    FROM orders
    ORDER BY "Customer ID", InvoiceDate
    LIMIT 30
""",conn)
print(q3)

    Customer ID  Invoice          InvoiceDate      next_order_date
0       12346.0   491725  2009-12-14 08:34:00  2009-12-14 11:00:00
1       12346.0   491742  2009-12-14 11:00:00  2009-12-14 11:02:00
2       12346.0   491744  2009-12-14 11:02:00  2009-12-18 10:47:00
3       12346.0   492718  2009-12-18 10:47:00  2009-12-18 10:55:00
4       12346.0   492722  2009-12-18 10:55:00  2010-01-04 09:24:00
5       12346.0   493410  2010-01-04 09:24:00  2010-01-04 09:53:00
6       12346.0   493412  2010-01-04 09:53:00  2010-01-14 13:50:00
7       12346.0   494450  2010-01-14 13:50:00  2010-01-22 13:30:00
8       12346.0   495295  2010-01-22 13:30:00  2010-01-26 17:27:00
9       12346.0  C495800  2010-01-26 17:27:00  2010-03-02 13:08:00
10      12346.0   499763  2010-03-02 13:08:00  2010-06-28 13:53:00
11      12346.0   513774  2010-06-28 13:53:00  2010-06-30 11:22:00
12      12346.0  C514024  2010-06-30 11:22:00  2010-10-04 09:54:00
13      12346.0  C525099  2010-10-04 09:54:00  2010-10-04 16:3

In [12]:
#04:Practical use case- kon se customers ke order-gaps sabse jyada hai(potential churners ka early signal):

In [13]:
q4 = pd.read_sql("""
    SELECT "Customer ID", AVG(days_gap) as avg_gap_between_orders
    FROM (
        SELECT "Customer ID",
                julianday(InvoiceDate) - julianday(LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate)) as days_gap
        FROM orders
    )
    WHERE days_gap IS NOT NULL
    GROUP BY "Customer ID"
    ORDER BY avg_gap_between_orders DESC
    LIMIT 15
""",conn)
print(q4)

    Customer ID  avg_gap_between_orders
0       14954.0              690.911111
1       16127.0              687.068750
2       14799.0              686.206944
3       16404.0              662.099306
4       17359.0              636.992361
5       14171.0              632.120139
6       17958.0              611.862500
7       12638.0              606.111806
8       17301.0              603.018750
9       16666.0              601.950000
10      18174.0              596.193750
11      16641.0              596.018750
12      17660.0              595.899306
13      15524.0              582.118056
14      16847.0              574.997917


In [14]:
#jin customers ka average gap bahut jyada hai(jaise 100+din), wo "irregular" buyers hai- Recency-based churn rule(Bouns day 35B me) isi tarah ke logic par based hoga.

In [15]:
#05:Month-over-month comparision(PDF ka original wording - "pichle month ke date ki compare karna"):

In [16]:
q5 = pd.read_sql("""
    SELECT year_month,  total_revenue,
           LAG(total_revenue) OVER (ORDER BY year_month) as previous_month_revenue,
           total_revenue - LAG(total_revenue) OVER (ORDER BY year_month) as revenue_change
    FROM (
        SELECT strftime('%Y-%m', InvoiceDate) as year_month, SUM(OrderValue) as total_revenue
        FROM orders
        GROUP BY year_month
    )
    ORDER BY year_month
""",conn)
print(q5)

   year_month  total_revenue  previous_month_revenue  revenue_change
0     2009-12     663272.050                     NaN             NaN
1     2010-01     531952.902              663272.050     -131319.148
2     2010-02     489399.586              531952.902      -42553.316
3     2010-03     635996.481              489399.586      146596.895
4     2010-04     560635.022              635996.481      -75361.459
5     2010-05     559924.550              560635.022        -710.472
6     2010-06     571459.910              559924.550       11535.360
7     2010-07     562785.900              571459.910       -8674.010
8     2010-08     587256.460              562785.900       24470.560
9     2010-09     781033.301              587256.460      193776.841
10    2010-10     964989.780              781033.301      183956.479
11    2010-11    1134879.282              964989.780      169889.502
12    2010-12     859227.370             1134879.282     -275651.912
13    2011-01     475074.380      

In [17]:
#yeh dikhata hai ki har month revenue pichle mahine se badha ya ghata.

In [18]:
#practice questions.

In [19]:
#1.Kaunse mahine mein sabse zyada revenue drop hua (revenue_change sabse negative)?

In [20]:
#

In [21]:
#2.Har customer ka sabse pehla aur sabse aakhri order date nikaalo (hint: MIN/MAX Window Function ya simple GROUP BY se), aur unke beech ka total gap (days) calculate karo — yeh "customer lifespan" hai, RFM analysis mein kaam aayega.

In [22]:
#

In [23]:
conn.close()